# Taller 6 — Agente con herramientas MCP: resumen y búsqueda de recursos relacionados

Este notebook construye un agente conversacional (LangChain + Ollama) que usa dos servidores **MCP** propios:

| Servidor | Puerto | Herramienta | Qué hace |
|---|---|---|---|
| `resumen_mcp_server.py` | 9081 | `resumir_archivo(ruta)` | Resume un `.txt` en un párrafo en español con `Qwen/Qwen2.5-1.5B-Instruct` |
| `similares_mcp_server.py` | 9082 | `buscar_similares(ruta, max_resultados)` | Extrae frases clave del texto (embeddings + MMR) y busca en internet páginas, artículos, blogs, noticias y PDF relacionados |

```
Usuario ──► Gradio ──► Agente LangChain (Ollama qwen2.5:3b)
                              │  MCP / streamable-http
                 ┌────────────┴─────────────┐
                 ▼                          ▼
      resumen_mcp_server :9081    similares_mcp_server :9082
```

**Requisitos**
- **Local:** Ollama instalado y corriendo. Las dependencias de `mcp_servers/requirements.txt` instaladas en el entorno del kernel.
- **Kaggle / Colab:** el notebook instala todo. En Kaggle activa **Settings → Internet on** y, si es posible, un acelerador **GPU**.
- La primera ejecución descarga unos 3.5 GB de modelos de Hugging Face y unos 2 GB del modelo de Ollama.

## 1. Configuración y detección del entorno

In [ ]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
IN_CLOUD = IN_COLAB or IN_KAGGLE

REPO_RAW_URL = 'https://raw.githubusercontent.com/jose-milciades/ICESI_NLP/main/mcp_servers'
SERVER_FILES = ['__init__.py', 'resumen_mcp_server.py', 'similares_mcp_server.py',
                'requirements.txt', 'ejemplo.txt']

OLLAMA_MODEL = 'qwen2.5:3b'
MCP_SERVERS = {  # script -> puerto
    'resumen_mcp_server.py': 9081,
    'similares_mcp_server.py': 9082,
}

if IN_KAGGLE:
    WORK_DIR = Path('/kaggle/working')
elif IN_COLAB:
    WORK_DIR = Path('/content')
else:
    WORK_DIR = Path.cwd()

entorno = 'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'
print(f'Entorno: {entorno} | directorio de trabajo: {WORK_DIR}')

## 2. Dependencias y código de los servidores MCP

En Kaggle/Colab se instalan las dependencias y se descargan los servidores desde GitHub (rama `main`). En local se usa la carpeta `mcp_servers/` del repositorio.

In [ ]:
import shutil
import subprocess
import urllib.request

if IN_CLOUD:
    SERVERS_DIR = WORK_DIR / 'mcp_servers'
    SERVERS_DIR.mkdir(parents=True, exist_ok=True)
    for name in SERVER_FILES:
        urllib.request.urlretrieve(f'{REPO_RAW_URL}/{name}', SERVERS_DIR / name)
    print(f'Servidores MCP descargados en {SERVERS_DIR}')

    print('Instalando dependencias de Python...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-U',
         '-r', str(SERVERS_DIR / 'requirements.txt'),
         'langchain', 'langchain-mcp-adapters', 'langchain-ollama', 'gradio'],
        check=True,
    )

    if shutil.which('ollama') is None:
        print('Instalando Ollama...')
        subprocess.run('apt-get install -y -qq zstd > /dev/null 2>&1; '
                       'curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1',
                       shell=True, check=True)
else:
    SERVERS_DIR = next((p for p in (WORK_DIR / 'mcp_servers', WORK_DIR.parent / 'mcp_servers')
                        if p.exists()), None)
    if SERVERS_DIR is None:
        raise FileNotFoundError('No se encontró la carpeta mcp_servers junto al notebook.')
    if shutil.which('ollama') is None:
        raise RuntimeError('Ollama no está instalado. Descárgalo de https://ollama.com/download')

for name in MCP_SERVERS:
    if not (SERVERS_DIR / name).exists():
        raise FileNotFoundError(f'Falta {name} en {SERVERS_DIR}')
print(f'Servidores MCP: {SERVERS_DIR}')

In [ ]:
# Código de los servidores (solo para revisión).
for name in MCP_SERVERS:
    path = SERVERS_DIR / name
    print(f'{path}\n' + '-' * 100)
    print(path.read_text(encoding='utf-8'))
    print('=' * 100 + '\n')

## 3. Inicio de Ollama y de los servidores MCP

La siguiente celda inicia en segundo plano los procesos que no estén corriendo, y espera a que cada uno abra su puerto. Los servidores tardan en arrancar porque cargan sus modelos; la primera vez además los descargan.

- Los logs quedan en la carpeta `logs/` (en local, en el directorio temporal del sistema).
- Si prefieres iniciarlos manualmente para ver su salida, ejecuta en terminales separadas `python mcp_servers/resumen_mcp_server.py` y `python mcp_servers/similares_mcp_server.py`; la celda detectará que ya están corriendo.
- Al terminar, ejecuta la última celda del notebook para apagarlos.

In [ ]:
import socket
import tempfile
import time

LOG_DIR = WORK_DIR / 'logs' if IN_CLOUD else Path(tempfile.gettempdir()) / 'mcp_taller6_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
BACKGROUND_PROCESSES = globals().get('BACKGROUND_PROCESSES', {})

def port_open(port: int) -> bool:
    with socket.socket() as sock:
        sock.settimeout(1)
        return sock.connect_ex(('localhost', port)) == 0

def start_background(name: str, command: list[str], port: int, timeout: int, cwd=None):
    if port_open(port):
        print(f'{name}: ya está corriendo en el puerto {port}')
        return
    log_path = LOG_DIR / f'{name}.log'
    print(f'{name}: iniciando (log en {log_path})...')
    # La salida va a un archivo: con subprocess.PIPE sin leer, el proceso se
    # bloquea cuando se llena el buffer.
    process = subprocess.Popen(command, cwd=cwd, stdout=open(log_path, 'w'),
                               stderr=subprocess.STDOUT)
    BACKGROUND_PROCESSES[name] = process
    start = time.time()
    while time.time() - start < timeout:
        if port_open(port):
            print(f'{name}: listo en el puerto {port} ({time.time() - start:.0f} s)')
            return
        if process.poll() is not None:
            raise RuntimeError(f'{name} terminó con error:\n{log_path.read_text()[-2000:]}')
        time.sleep(3)
    raise TimeoutError(f'{name} no respondió en {timeout} s. Revisa {log_path}')

start_background('ollama', ['ollama', 'serve'], 11434, timeout=60)
print(f'Descargando {OLLAMA_MODEL} en Ollama si hace falta...')
subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for script, port in MCP_SERVERS.items():
    start_background(script.removesuffix('.py'), [sys.executable, script], port,
                     timeout=900, cwd=SERVERS_DIR)

## 4. Cliente MCP y agente

`MultiServerMCPClient` se conecta a ambos servidores por HTTP y convierte sus herramientas en herramientas de LangChain. El agente (`create_agent`) decide cuándo invocarlas. Además se definen utilidades para invocarlas directamente (modo determinista) y para redactar la respuesta final a partir de su salida.

In [ ]:
import re
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)

# Ambas herramientas tardan ~20-30 s, por eso se amplía el timeout de lectura.
client = MultiServerMCPClient(
    {
        'resumen': {
            'transport': 'http',
            'url': 'http://localhost:9081/mcp',
            'sse_read_timeout': 600,
        },
        'similares': {
            'transport': 'http',
            'url': 'http://localhost:9082/mcp',
            'sse_read_timeout': 600,
        },
    }
)

async def load_mcp_tools_with_diagnostics(mcp_client):
    try:
        return await mcp_client.get_tools()
    except Exception as exc:
        raise RuntimeError(
            'No pude conectar con los servidores MCP. Ejecuta la celda de inicio de '
            'servidores y verifica que los puertos 9081 y 9082 respondan.'
        ) from exc

DEFAULT_FILE = 'ejemplo.txt'

AGENT_SYSTEM_PROMPT = (
    'Eres un asistente que responde en español. Tienes dos herramientas: '
    '`resumir_archivo` resume el contenido de un archivo de texto y '
    '`buscar_similares` busca en internet recursos (páginas, artículos, blogs, noticias, PDF) '
    'relacionados con el contenido de un archivo. Ambas reciben la ruta del archivo; '
    f'si el usuario no indica una, usa "{DEFAULT_FILE}". '
    'Usa las herramientas en lugar de inventar contenido y, al listar recursos, '
    'copia las URL exactamente como las devuelve la herramienta.'
)

mcp_tools = await load_mcp_tools_with_diagnostics(client)
agent_mcp = create_agent(model=llm, tools=mcp_tools, system_prompt=AGENT_SYSTEM_PROMPT)

def available_tool_names():
    return [tool.name for tool in mcp_tools]

def tool_from_name(tool_name: str):
    return next((tool for tool in mcp_tools if tool.name == tool_name), None)

def detect_suggested_tool(question: str) -> str | None:
    lower = question.lower()
    search_signals = ['relacionad', 'similar', 'buscar', 'busca', 'internet', 'url',
                      'enlace', 'link', 'blog', 'articulo', 'artículo', 'pdf', 'fuente']
    if any(word in lower for word in search_signals):
        return 'buscar_similares'
    summary_signals = ['resum', 'idea general', 'idea principal', 'de que trata',
                       'de qué trata', 'sintetiza', 'sintesis', 'síntesis']
    if any(word in lower for word in summary_signals):
        return 'resumir_archivo'
    return None

def select_tool_name(question: str, preferred_tool: str | None):
    if preferred_tool and preferred_tool != 'Auto':
        return preferred_tool
    return detect_suggested_tool(question)

def infer_file_path(question: str) -> str:
    # Primera ruta .txt mencionada en el mensaje. Las rutas relativas las
    # resuelve el servidor respecto a la carpeta mcp_servers/.
    match = re.search(r"([\w\-./~]+\.txt)\b", question, flags=re.IGNORECASE)
    return match.group(1) if match else DEFAULT_FILE

def infer_max_results(question: str, default: int = 10) -> int:
    # Ej.: "busca 5 artículos relacionados con ejemplo.txt".
    match = re.search(r"\b(\d{1,2})\s+(?:resultados|recursos|articulos|artículos|enlaces|links|urls|fuentes|blogs)",
                      question, flags=re.IGNORECASE)
    return int(match.group(1)) if match else default

def is_ollama_connection_error(exc: Exception) -> bool:
    text = str(exc).lower()
    signals = ['all connection attempts failed', 'connection refused', 'failed to connect',
               'connecterror', 'could not connect']
    return any(signal in text for signal in signals)

def extract_text_content(content) -> str:
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        parts = [extract_text_content(item) for item in content]
        return '\n'.join(part for part in parts if part)
    if isinstance(content, dict):
        for key in ('text', 'content', 'result'):
            if content.get(key) is not None:
                return extract_text_content(content[key])
        return str(content)
    for attr in ('text', 'content', 'result'):
        value = getattr(content, attr, None)
        if value is not None:
            return extract_text_content(value)
    return str(content)

def fallback_tool_answer(tool_name: str, tool_output, llm_error: str | None = None) -> str:
    text = extract_text_content(tool_output)
    if llm_error:
        return (f'Resultado directo de la herramienta {tool_name}:\n{text}\n\n'
                f'Nota: no pude contactar Ollama para redactar una respuesta ({llm_error}).')
    return f'Resultado directo de la herramienta {tool_name}:\n{text}'

async def run_selected_tool(tool_name: str, question: str, file_path: str | None = None):
    tool = tool_from_name(tool_name)
    if tool is None:
        raise ValueError(f'Herramienta no encontrada: {tool_name}')
    ruta = file_path or infer_file_path(question)
    if tool_name == 'resumir_archivo':
        payload = {'ruta': ruta}
    elif tool_name == 'buscar_similares':
        payload = {'ruta': ruta, 'max_resultados': infer_max_results(question)}
    else:
        raise ValueError(f'Herramienta no soportada: {tool_name}')
    return await tool.ainvoke(payload)

def tool_message_used(messages) -> bool:
    return any(getattr(msg, 'type', '') == 'tool' for msg in messages)

SYNTHESIS_INSTRUCTIONS = {
    # El resumen ya viene redactado por el servidor: solo se ajusta a la pregunta.
    'resumir_archivo': 'Responde en español usando solamente el resumen entregado por la herramienta. '
                       'No agregues datos que no estén en el resumen.',
    # Para la búsqueda se conservan las URL tal cual para que sean utilizables.
    'buscar_similares': 'La herramienta buscó en internet recursos relacionados con el CONTENIDO del '
                        'archivo (no con su nombre). Responde en español: indica en una frase el tema '
                        'detectado y presenta los recursos como una lista con título, tipo (archivo, blog, '
                        'wikipedia, artículo, noticia, video) y la URL copiada exactamente igual. '
                        'No inventes recursos, no modifiques las URL y no comentes el nombre del archivo.',
}

async def synthesize_from_tool(question: str, tool_name: str, tool_output):
    try:
        system_prompt = SYNTHESIS_INSTRUCTIONS.get(
            tool_name, 'Responde en español, de forma breve, usando solo el resultado de la herramienta.')
        synthesis = await llm.ainvoke([
            ('system', system_prompt),
            ('user', f'Pregunta: {question}\n\nSalida de {tool_name}:\n'
                     f'{extract_text_content(tool_output)}\n\nRedacta una respuesta final clara.'),
        ])
        return extract_text_content(synthesis.content)
    except Exception as exc:
        if is_ollama_connection_error(exc):
            return fallback_tool_answer(tool_name, tool_output, llm_error=str(exc))
        raise

print('Herramientas MCP disponibles:', available_tool_names())

## 5. Prueba rápida de las herramientas

Antes de abrir la interfaz, se invoca cada herramienta directamente y luego el agente, para comprobar que todo funciona de punta a punta. Cada llamada tarda entre 20 y 30 s.

In [ ]:
print('Archivo de prueba:', (SERVERS_DIR / DEFAULT_FILE).read_text(encoding='utf-8')[:300], '...\n')

resumen = await run_selected_tool('resumir_archivo', f'Resume {DEFAULT_FILE}')
print('=== resumir_archivo ===')
print(extract_text_content(resumen))

similares = await run_selected_tool('buscar_similares', f'Busca 5 recursos relacionados con {DEFAULT_FILE}')
print('\n=== buscar_similares ===')
print(extract_text_content(similares))

In [ ]:
pregunta = f'¿De qué trata el archivo {DEFAULT_FILE}? Dame la idea general.'
result = await agent_mcp.ainvoke({'messages': [('user', pregunta)]})
print('¿El agente usó una herramienta?', tool_message_used(result['messages']))
print(extract_text_content(result['messages'][-1].content))

## 6. Interfaz de chat (Gradio)

Modos de ejecución:
- **Guided:** el agente decide, pero se le sugiere la herramienta detectada en la pregunta. Si no la usa y el *fallback* está activo, se invoca directamente.
- **Autonomous:** el agente decide solo qué herramienta usar.
- **Deterministic:** se invoca directamente la herramienta detectada (o la elegida) y el LLM solo redacta la respuesta.

Puedes **subir tu propio archivo `.txt`**. Si no subes ninguno, se usa el que menciones en la pregunta o `ejemplo.txt`. En Kaggle/Colab la interfaz se abre con un enlace público (`share=True`).

In [ ]:
import inspect
import gradio as gr

def history_to_messages(question, chat_history):
    messages = []
    for item in (chat_history or []):
        role = item.get('role')
        content = item.get('content')
        if role in {'user', 'assistant'} and content is not None:
            messages.append((role, extract_text_content(content)))
    messages.append(('user', question))
    return messages

def compact_trace_text(trace: dict):
    lines = [f"{key}={trace.get(key)}" for key in
             ('mode', 'tool_selected', 'file', 'used_tool', 'guided_fallback_used')]
    if trace.get('tool_error'):
        lines.append(f"tool_error={trace['tool_error']}")
    return ' | '.join(lines)

async def recover_with_direct_tool_if_possible(question, selected_tool, file_path, trace, llm_exc):
    if selected_tool is None:
        return None
    try:
        tool_output = await run_selected_tool(selected_tool, question, file_path)
        trace['used_tool'] = True
        trace['guided_fallback_used'] = True
        return fallback_tool_answer(selected_tool, tool_output, llm_error=str(llm_exc))
    except Exception:
        return None

async def respond(question, chat_history, uploaded_file, mode, preferred_tool, guided_fallback, show_trace):
    chat_history = chat_history or []
    if not question or not question.strip():
        return '', chat_history
    mode = mode or 'Guided'
    file_path = uploaded_file if uploaded_file else infer_file_path(question)
    selected_tool = select_tool_name(question, preferred_tool)

    # Para el agente, se explicita la ruta del archivo a usar en el mensaje.
    agent_question = f'{question}\n\n(Archivo a usar: {file_path})' if uploaded_file else question
    messages = history_to_messages(agent_question, chat_history)
    trace = {'mode': mode, 'tool_selected': selected_tool, 'file': file_path,
             'used_tool': False, 'guided_fallback_used': False, 'tool_error': None}

    try:
        if mode == 'Deterministic':
            if selected_tool is None:
                answer = ('No pude inferir qué herramienta usar. Pide un resumen o una búsqueda de '
                          'recursos relacionados, o elige la herramienta en "Tool preferida".')
            else:
                tool_output = await run_selected_tool(selected_tool, question, file_path)
                answer = await synthesize_from_tool(question, selected_tool, tool_output)
                trace['used_tool'] = True

        elif mode == 'Guided':
            guided_messages = messages
            if selected_tool is not None:
                guided_messages = [(
                    'system',
                    f'Para responder usa la herramienta {selected_tool} con la ruta "{file_path}".',
                )] + messages
            result = await agent_mcp.ainvoke({'messages': guided_messages})
            used_tool = tool_message_used(result['messages'])
            answer = extract_text_content(result['messages'][-1].content)
            trace['used_tool'] = used_tool

            if guided_fallback and not used_tool and selected_tool is not None:
                tool_output = await run_selected_tool(selected_tool, question, file_path)
                answer = await synthesize_from_tool(question, selected_tool, tool_output)
                trace['used_tool'] = True
                trace['guided_fallback_used'] = True

        else:  # Autonomous
            result = await agent_mcp.ainvoke({'messages': messages})
            trace['used_tool'] = tool_message_used(result['messages'])
            answer = extract_text_content(result['messages'][-1].content)

    except Exception as exc:
        trace['tool_error'] = str(exc)
        if is_ollama_connection_error(exc):
            recovered = await recover_with_direct_tool_if_possible(
                question, selected_tool, file_path, trace, exc)
            answer = recovered or ('No pude conectar con Ollama. Inicia Ollama con `ollama serve` '
                                   'o usa el modo Deterministic.')
        else:
            answer = f'Ocurrió un error: {exc}'

    if show_trace:
        answer = f'{answer}\n\n---\nTraza MCP: {compact_trace_text(trace)}'

    updated_history = chat_history + [
        {'role': 'user', 'content': question},
        {'role': 'assistant', 'content': answer},
    ]
    return '', updated_history

def reset_chat():
    return '', []

In [ ]:
# Prueba de la lógica del chat sin abrir la interfaz (modo Deterministic).
_, historial = await respond(f'Busca 3 recursos relacionados con {DEFAULT_FILE}', [], None,
                             'Deterministic', 'Auto', True, True)
print(historial[-1]['content'])

In [ ]:
# Gradio 6 usa el formato de mensajes por defecto; en Gradio 4/5 hay que pedirlo.
chatbot_kwargs = ({'type': 'messages'}
                  if 'type' in inspect.signature(gr.Chatbot.__init__).parameters else {})

with gr.Blocks(title='Agente MCP: resumen y recursos relacionados') as gr_blocks:
    gr.Markdown('## Agente MCP: resumen de textos y búsqueda de recursos relacionados')

    with gr.Row():
        mode = gr.Dropdown(choices=['Guided', 'Autonomous', 'Deterministic'],
                           value='Guided', label='Modo de ejecución')
        preferred_tool = gr.Dropdown(choices=['Auto', 'resumir_archivo', 'buscar_similares'],
                                     value='Auto', label='Tool preferida')
    with gr.Row():
        guided_fallback = gr.Checkbox(
            label='En modo Guided, usar fallback de tool si el agente no la invoca', value=True)
        show_trace = gr.Checkbox(label='Mostrar traza de uso MCP', value=False)

    uploaded_file = gr.File(label='Archivo .txt (opcional; por defecto ejemplo.txt)',
                            file_types=['.txt'], type='filepath')
    chatbot = gr.Chatbot(label='Historial', **chatbot_kwargs)
    msg = gr.Textbox(
        label='¿Qué quieres hacer?',
        placeholder='Ejemplo: Resume ejemplo.txt  |  Busca 5 artículos relacionados con ejemplo.txt',
    )
    clear = gr.Button('Limpiar')

    msg.submit(respond,
               [msg, chatbot, uploaded_file, mode, preferred_tool, guided_fallback, show_trace],
               [msg, chatbot])
    clear.click(reset_chat, None, [msg, chatbot], queue=False)

# En Kaggle/Colab no hay acceso a localhost desde el navegador: se usa un enlace público.
gr_blocks.launch(share=IN_CLOUD, inline=False)

## 7. Apagado

Cierra la interfaz y detiene los procesos que inició este notebook. Los que hayas iniciado manualmente en otra terminal se detienen con `Ctrl+C`.

In [ ]:
gr_blocks.close()

for name, process in BACKGROUND_PROCESSES.items():
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
        print(f'{name}: detenido')
BACKGROUND_PROCESSES.clear()